In [1]:
import numpy as np
import pennylane as qml
import simulator

/Users/adam-ukj7r05xnu2fywx/miniconda3/envs/NoisyCircuits/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [11]:
num_qubits = 8

@qml.qnode(qml.device("lightning.qubit", wires=num_qubits))
def circuit(a):
    global instruction_list
    for i in range(num_qubits):
        qml.Hadamard(wires=i)
        instruction_list.append(["h", [i, i], 0.0])
    for d in range(len(a)//num_qubits):
        for i in range(num_qubits):
            qml.RY(a[d+i], wires=i)
            qml.RX(a[d+i], wires=i)
            qml.PhaseShift(a[d+i], wires=i)
            instruction_list.append(["ry", [i, i], a[d+i]])
            instruction_list.append(["rx", [i, i], a[d+i]])
            instruction_list.append(["p", [i, i], a[d+i]])
        for i in range(num_qubits-1):
            qml.SWAP(wires=[i, i+1])
            instruction_list.append(["swap", [i, i+1], 0.0])
    for i in range(num_qubits):
        qml.Hadamard(wires=i)
        instruction_list.append(["h", [i, i], 0.0])
    for i in range(num_qubits-1):
        qml.CNOT(wires=[i, i+1])
        instruction_list.append(["cx", [i, i+1], 0.0])
    return qml.state()

In [13]:
instruction_list = []
a = np.random.rand(45*num_qubits)
state_ref = circuit(a)
state_sim = np.zeros_like(state_ref)
simulator.simulate_circuit(instruction_list, state_sim, {}, {}, num_qubits, False, 1, True, 1)
state_sim = state_sim.reshape([2]*num_qubits).transpose(list(range(num_qubits))[::-1]).reshape(-1)
print("State fidelity: {:.5f}".format(np.vdot(state_ref, state_sim).real))

State fidelity: 1.00000
